# 06 - Evaluation & Visualization

Generate all charts for the report and viva presentation:
1. Model comparison bar chart (accuracy, F1, precision, recall) with error bars
2. Accuracy vs. inference speed scatter plot
3. Learning curves (25%, 50%, 100% training data)
4. Confusion matrices
5. Error type distribution

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
from src.results import collect_results

sns.set_theme(style='whitegrid', font_scale=1.1)
COLORS = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
RESULTS_DIR = '../results'
PLOT_DIR = '../plots'
os.makedirs(PLOT_DIR, exist_ok=True)

## Load Results

Reads everything notebooks 02-05 (or `python main.py`) saved under `results/<model>/`.
Models that haven't been run yet are simply left out of the charts.

In [ ]:
comparison, curves, cm_dict, error_counts = collect_results(RESULTS_DIR)
assert comparison, 'No results found - run notebooks 02-05 (or main.py) first.'

rows = []
for model, r in comparison.items():
    row = {'Model': model}
    for m in ['accuracy', 'f1', 'precision', 'recall']:
        row[m.capitalize()] = r[m]
        row[f'{m.capitalize()}_std'] = r[f'{m}_std']
    row['Speed (samples/s)'] = r.get('samples_per_second', 0)
    rows.append(row)

results = pd.DataFrame(rows)
results

## 1. Model Comparison Bar Chart (with Error Bars)

In [ ]:
metrics_to_plot = ['Accuracy', 'F1', 'Precision', 'Recall']
models = results['Model'].tolist()
x = np.arange(len(metrics_to_plot))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 6))

for i, (_, row) in enumerate(results.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    errors = [row[f'{m}_std'] for m in metrics_to_plot]
    offset = (i - len(models)/2 + 0.5) * width
    bars = ax.bar(
        x + offset, values, width,
        label=row['Model'], color=COLORS[i % len(COLORS)],
        yerr=errors, capsize=3, alpha=0.85
    )
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold'
        )

ax.set_ylabel('Score')
ax.set_title('Model Comparison: Bangla Toxic Comment Detection')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 2. Accuracy vs. Inference Speed

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Speed is only known for models evaluated with build_evaluation (timed inference)
timed = results[results['Speed (samples/s)'] > 0].reset_index(drop=True)
for i, (_, row) in enumerate(timed.iterrows()):
    ax.scatter(
        row['Speed (samples/s)'], row['Accuracy'],
        s=200, c=COLORS[i % len(COLORS)], label=row['Model'],
        edgecolors='black', linewidth=1, zorder=5
    )
    ax.annotate(
        row['Model'],
        (row['Speed (samples/s)'], row['Accuracy']),
        textcoords='offset points', xytext=(10, 10),
        fontsize=10, fontweight='bold'
    )

ax.set_xlabel('Inference Speed (samples/second)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy vs. Inference Speed', fontsize=14)
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/accuracy_vs_speed.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Learning Curves (25%, 50%, 100%)

In [ ]:
lc_data = {
    model: {
        'fractions': [d['fraction'] for d in data],
        'accuracy': [d['eval_accuracy'] for d in data],
        'f1': [d['eval_f1'] for d in data],
    }
    for model, data in curves.items()
}

if not lc_data:
    print('No learning curves found - run notebooks 03/04 first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    all_vals = [v for d in lc_data.values() for m in ('accuracy', 'f1') for v in d[m]]

    for metric_idx, metric in enumerate(['accuracy', 'f1']):
        ax = axes[metric_idx]
        for i, (model_name, data) in enumerate(lc_data.items()):
            fraction_labels = [f'{fr * 100:.0f}%' for fr in data['fractions']]
            ax.plot(
                fraction_labels, data[metric],
                marker='o', linewidth=2, markersize=8,
                label=model_name, color=COLORS[(i + 1) % len(COLORS)]
            )
            for j, val in enumerate(data[metric]):
                ax.annotate(f'{val:.2f}', (fraction_labels[j], val),
                           textcoords='offset points', xytext=(0, 10),
                           ha='center', fontsize=9)

        ax.set_xlabel('Training Data Fraction')
        ax.set_ylabel(metric.upper())
        ax.set_title(f'Learning Curve - {metric.upper()}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(max(0, min(all_vals) - 0.1), 1.0)

    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/learning_curves.png', dpi=200, bbox_inches='tight')
    plt.show()

## 4. Confusion Matrices

In [ ]:
n = len(cm_dict)
ncols = min(n, 2)
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

for ax, (model_name, cm) in zip(axes.flat, cm_dict.items()):
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Non-Toxic', 'Toxic'],
        yticklabels=['Non-Toxic', 'Toxic'],
        ax=ax
    )
    ax.set_title(f'{model_name}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

for ax in list(axes.flat)[n:]:
    ax.axis('off')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/confusion_matrices.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Error Type Distribution

In [ ]:
error_data = pd.DataFrame({
    'Model': list(error_counts),
    'False Positives': [e['false_positive'] for e in error_counts.values()],
    'False Negatives': [e['false_negative'] for e in error_counts.values()],
})

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(error_data))
width = 0.35

bars1 = ax.bar(x - width/2, error_data['False Positives'], width,
               label='False Positives (non-toxic -> toxic)', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, error_data['False Negatives'], width,
               label='False Negatives (toxic -> non-toxic)', color='#f39c12', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Error Count')
ax.set_title('Error Type Distribution by Model')
ax.set_xticks(x)
ax.set_xticklabels(error_data['Model'], rotation=15)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 3,
                str(int(h)), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/error_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## Summary Table for Report

In [ ]:
summary = results[['Model', 'Accuracy', 'F1', 'Precision', 'Recall', 'Speed (samples/s)']].copy()
summary['Accuracy'] = summary.apply(
    lambda r: f"{r['Accuracy']:.4f} +/- {results.loc[r.name, 'Accuracy_std']:.4f}"
    if results.loc[r.name, 'Accuracy_std'] > 0 else f"{r['Accuracy']:.4f}", axis=1
)
summary['F1'] = summary.apply(
    lambda r: f"{results.loc[r.name, 'F1']:.4f} +/- {results.loc[r.name, 'F1_std']:.4f}"
    if results.loc[r.name, 'F1_std'] > 0 else f"{results.loc[r.name, 'F1']:.4f}", axis=1
)
print(summary.to_markdown(index=False))